# Volumetric Functions

Version 1.0
Neil Graham and Greg Scott

Updated 21/5/2018

Still to do: VBM analysis functionality

Pre-requisites:

- this is designed to run on the c3nl sample data
- you should have run the c3nl_running_commands notebook first in order to convert the dicoms to nifties

Scripts needed for it to work:
- hpcwrapmatlab.sh
- segment_t1.m
- make_template.m
- generate_flowfields.m
- move_to_mni.m


In [ ]:
!DATADIR=$WORK/sampledata/c3nl-example-data-raw
setenv('PATH', [ getenv('PATH'),':/apps/fsl/5.0.8/fsl/bin']);
setenv('FSLDIR', '/apps/fsl/5.0.8/fsl');
setenv('FSLOUTPUTTYPE', 'NIFTI_GZ');
!NOTEBOOKDIR=`pwd`
!echo $NOTEBOOKDIR;

In [ ]:
%%shell
module load fsl

### Stage 1: Simple calcuation of volumes from T1s
### File management - create a preprocessed directory

In [ ]:
%%shell
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            for modality in `ls ${DATADIR}/raw/${subject}/${visit}`
            do
                echo ${subject}/${visit}/${modality}
                mkdir -p ${DATADIR}/proc/${subject}/${visit}/${modality}/nifti
            done;
        done
    done;

### Segment everyone's T1 from each visit separately

This step uses SPM12s segment function to separate grey matter, white matter and CSF which are called C1,C2 and C3 respectively. It also outputs some rigidly aligned segmented files called RC1, RC2 and RC3 which are useful later if you need to normalise the images to standard, eg. MNI space, via SPM's tool 'dartel'.

In [ ]:
%%shell
    echo -n "" > commands.txt
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            echo ${subject}/${visit}
            cp ${DATADIR}/raw/${subject}/${visit}/t1/nifti/${subject}_${visit}_t1.nii ${DATADIR}/proc/${subject}/${visit}/t1/nifti
            echo "hpcwrapmatlab.sh \"maxNumCompThreads(3); segment_t1('${DATADIR}/proc/${subject}/${visit}/t1/nifti/${subject}_${visit}_t1.nii');\"" >> commands.txt
        done
    done
    ./hpcrunarrayjob.sh commands.txt 01:00:00 3 6Gb

Great news. You now have a textfile in each timepoint's folder with the results of the volume calculations.
You can tidy them into one beautiful CSV with the next cell.

### Make a big spreadsheet (CSV) of volume results


In [ ]:
%%shell
    echo "subject,volume,file,gm,wm,csf" > volumes.csv
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            file="${DATADIR}/proc/${subject}/${visit}/t1/nifti/*_vols.txt";
            
            if [ -f ${file} ]
            then
                echo -n "${subject},${visit}," >> volumes.csv;
                tail -n 1 ${file} >> volumes.csv;
            fi
        done
    done

### Let's have a look!

Or alternatively, if you like you can right click and 'download' from the file list on the left side of the screen, and load it in excel. Or an even fancier program if you like. You could even do some stats in an 'R' jupyter notebook.

In [ ]:
%%shell
cat volumes.csv

## Stage 2 - Optional - Moving the images to standard space, to be able to do statistics...

2.1 - Make a study specific templte

Rather than going directly to MNI space the SPM dartel approach to registration goes through an intermediate study specific template. This is thought to result in better registration.

List the subject IDs that you want to put into the standard space here. The script will make you a nice subjects.txt
You might want to have equal numbers of patients and controls making the template for example (if you have uneven numbers)

In [ ]:
%%file subjects_for_template.txt
con_001
con_002
pat_001

In [ ]:
%%shell

# this uses the RC rigidly aligned files from the segmentation output : if you want to have a look at them use this in bash
# !find ${DATADIR} -type f -wholename "*rc*nii"


    files=""
    visit="v1"
    for subject in `cat subjects_for_template.txt`
    do
         rc1="${DATADIR}/proc/${subject}/${visit}/t1/nifti/rc1${subject}_${visit}_t1.nii";
         rc2="${DATADIR}/proc/${subject}/${visit}/t1/nifti/rc2${subject}_${visit}_t1.nii";
         rc3="${DATADIR}/proc/${subject}/${visit}/t1/nifti/rc3${subject}_${visit}_t1.nii";
         
         if [ -z "${files}" ]; then
             files="'${rc1}','${rc2}','${rc3}'";
         else
             files="${files},'${rc1}','${rc2}','${rc3}'";
         fi
    done;
     echo "hpcwrapmatlab.sh \"maxNumCompThreads(3); make_template('Template_${visit}', ${files})\"" >> commands2.txt;
     
     
    ./hpcrunarrayjob.sh commands2.txt 01:00:00 3 6Gb

In [ ]:
%check on your job
!qstat

Move the files to a new folder for your study specific template

In [ ]:
%%shell
# when the job is done, you need to move the completed template files to a nice new folder, as by default they are dumped into the first subject's folder 

visit="v1"
subject=$(head -n 1 subjects_for_template.txt)
echo $subject;
mkdir ${DATADIR}/proc/templates/;
mkdir ${DATADIR}/proc/templates/${visit};
mv ${DATADIR}/proc/${subject}/${visit}/t1/nifti/Template_v1_* ${DATADIR}/proc/templates/${visit}/;
ls ${DATADIR}/proc/templates/${visit};

2.2.  Generate flowfields to study template

Having made the template, you need to generate flowfields from each subject's t1 to this template

In [ ]:
%%shell
   
   template_visit="v1"
   template_basename="Template"
   template=${DATADIR}/proc/templates/${template_visit}/${template_basename}_${template_visit}

    unset files;
    echo "-n" > commands4.txt;
    
    
    #  Use this and a text file with subject names if you don't want to make flowfields for EVERYONE!
    # for subject in `cat subjects_to_align.txt`
    
    
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
           
             files=""
           

             rc1="${DATADIR}/proc/${subject}/${visit}/t1/nifti/rc1${subject}_${visit}_t1.nii";
             rc2="${DATADIR}/proc/${subject}/${visit}/t1/nifti/rc2${subject}_${visit}_t1.nii";
             rc3="${DATADIR}/proc/${subject}/${visit}/t1/nifti/rc3${subject}_${visit}_t1.nii";
            
         
             files="'${rc1}','${rc2}','${rc3}'";
          
              
             #echo hpcwrapmatlab.sh "generate_flowfields('${template}', ${files});" 
             
             
             if [ -f $rc1 ];
             then
             
            
             ######### Uncomment this section if you want to run it directly in notebook rather than using the PBS queue submission system
             #bash hpcwrapmatlab.sh "generate_flowfields('${template}', ${files});" 
             ##########################
             
             ##### Comment out this section if you want to run directly
             echo "hpcwrapmatlab.sh \"generate_flowfields('${template}', ${files})\"" >> commands4.txt;
             #########
             
             
             else
             echo "No rc1 file for ${subject} at visit ${visit}";
             fi
             unset files;
           
           
           done
    done
   
   ####### Comment out this section if you want to run directly
   ./hpcrunarrayjob.sh commands4.txt 01:00:00 3 6Gb
   ###############

    


In [ ]:
%check on your job
!qstat

### Use the newly generated flowfields to send your images to MNI space

In [ ]:
%%shell
   
   template_visit="v1"
   template_basename="Template"
   template=${DATADIR}/proc/templates/${template_visit}/${template_basename}_${template_visit}_6.nii
  #  echo template path is ${template}
    unset files;
    echo "-n" > commands5.txt;
   
   
   #for subject in `cat subjects_to_align.txt`
   for subject in `ls ${DATADIR}/raw`
   do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            
            files=""
           
            
             
             u_rc1="${DATADIR}/proc/${subject}/${visit}/t1/nifti/u_rc1${subject}_${visit}_t1.nii";
             c1="${DATADIR}/proc/${subject}/${visit}/t1/nifti/c1${subject}_${visit}_t1.nii";
             c2="${DATADIR}/proc/${subject}/${visit}/t1/nifti/c2${subject}_${visit}_t1.nii";
             c3="${DATADIR}/proc/${subject}/${visit}/t1/nifti/c3${subject}_${visit}_t1.nii";
         
             files="'${u_rc1}','${c1}','${c2}','${c3}'";
          
              
             
          #   echo hpcwrapmatlab.sh "move_to_mni('${template}', ${files});" 
             
             ######### Uncomment this section if you want to run it directly rather than using the PBS queue submission system
             
             if [ -f $u_rc1 ];
             then
             echo bash hpcwrapmatlab.sh "move_to_mni('${template}', ${files});" 
             ##########################
             
             ##### Comment out this section if you want to run directly
             echo "hpcwrapmatlab.sh \"move_to_mni('${template}', ${files})\"" >> commands5.txt;
             #########
            
            else
            echo "No flowfield for this person ${subject} and timepoint ${visit}";
            fi
            
            unset files;
         done;
        
    done
   
   ####### Comment out this section if you want to run directly
   ./hpcrunarrayjob.sh commands5.txt 01:00:00 3 6Gb
   ###############




Congratulations! You might now want to do some statistical comparisons, such as between your groups. Have a look at http://courses.c3nl.com for more information on using tools to do this, such as fsl's randomise (a more popular choice than SPM)